# Structs

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Group related values in a struct, give it a name with typedef, and use arrays of structs.
- Pass structs to functions by value or through a pointer with ->, and explain what copying a struct copies.
- Name a fixed set of choices with an enum and switch on it.

**In this notebook**, one idea at a time (run each cell before moving on):
1. A struct groups values
2. Assigning copies every field
3. Structs and functions
4. Arrays of structs, and arrays inside structs
5. enum: naming a set of choices
6. sizeof and padding
7. Comparing structs

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. A struct groups values

A **struct** groups values of different types under one name, like a Java class with only fields. `typedef` gives the type a short name, so you write `Point` instead of `struct Point`. Create one with the values in braces, in order, or by field name (`.y = 4`), and reach the fields with a dot.

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

int main(void) {
    Point a = {3, 4};
    Point b = {.y = 10, .x = -1};
    a.x = a.x + 1;
    printf("a = (%d, %d), b = (%d, %d)\n", a.x, a.y, b.x, b.y);
    return 0;
}

🧪 **Try it:** add a field `char label` to the struct, give it a value in both points and print it.

## 2. Assigning copies every field

`Point b = a;` **copies every field**: `b` is a separate struct, and changing it leaves `a` alone. This is different from Java objects, where assigning shares one object. To work on the original, use a pointer. The slides show both cases.

#### 🎞️ Struct copies and pointers

![Point b = a copies the fields, so changing b leaves a alone; a pointer p = &a changes a itself](struct-copy-steps.svg)

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

int main(void) {
    Point a = {3, 4};
    Point b = a;          // a copy
    b.x = 9;
    Point *p = &a;        // a pointer to the original
    p->x = 7;             // p->x means (*p).x
    printf("a = (%d, %d), b = (%d, %d)\n", a.x, a.y, b.x, b.y);
    return 0;
}

🧪 **Try it:** predict both points before running. Then write `(*p).y = 0;` and check it does the same as `p->y = 0;`.

## 3. Structs and functions

Passing a struct to a function **copies** it, which is fine for reading a small struct. When the function must **change** the caller's struct, or the struct is big, pass a pointer (`Rect *r`) and use `r->field`. Add `const` when the function only reads through the pointer.

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

void move_copy(Point p, int dx) {     // gets a copy
    p.x += dx;
}

void move(Point *p, int dx) {         // gets the address
    p->x += dx;
}

int main(void) {
    Point a = {0, 0};
    move_copy(a, 5);
    printf("after move_copy: %d\n", a.x);
    move(&a, 5);
    printf("after move: %d\n", a.x);
    return 0;
}

🧪 **Try it:** make `move_copy` **return** the moved point (`Point move_copy(...)`) and use it as `a = move_copy(a, 5);`: another way to get the result out.

### ✏️ Exercise: A rectangle's area

Return the area of a Rect passed by value: the function gets a copy, which is fine for reading.

In [ ]:
typedef struct {
    int w;
    int h;
} Rect;

int rect_area(Rect r) {
    // TODO: multiply the fields
    return 0;
}

int main(void) {
    Rect r = {3, 4};
    check_int("3 by 4", rect_area(r), 12);
    check_int("a line", rect_area((Rect) {5, 0}), 0);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
typedef struct {
    int w;
    int h;
} Rect;

int rect_area(Rect r) {
    return r.w * r.h;
}

int main(void) {
    Rect r = {3, 4};
    check_int("3 by 4", rect_area(r), 12);
    check_int("a line", rect_area((Rect) {5, 0}), 0);
    return 0;
}
```

</details>

### ✏️ Exercise: Scale a rectangle

Multiply both sides by k, changing the caller's Rect. That takes a pointer and `->`.

In [ ]:
typedef struct {
    int w;
    int h;
} Rect;

void scale(Rect *r, int k) {
    // TODO: change r->w and r->h
}

int main(void) {
    Rect r = {3, 4};
    scale(&r, 2);
    check_int("w doubled", r.w, 6);
    check_int("h doubled", r.h, 8);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
typedef struct {
    int w;
    int h;
} Rect;

void scale(Rect *r, int k) {
    r->w *= k;
    r->h *= k;
}

int main(void) {
    Rect r = {3, 4};
    scale(&r, 2);
    check_int("w doubled", r.w, 6);
    check_int("h doubled", r.h, 8);
    return 0;
}
```

</details>

## 4. Arrays of structs, and arrays inside structs

Structs combine with arrays both ways. An **array of structs** is a table: `accounts[1].balance`. A struct can hold an **array**, such as a fixed-size name: `char owner[21]`. Copy text into it with `snprintf` or `strcpy` (after checking it fits), never with `=`.

In [ ]:
#include <stdio.h>

typedef struct {
    char name[16];
    int score;
} Player;

int main(void) {
    Player team[3] = {{"Ada", 12}, {"Alan", 30}, {"Grace", 21}};
    snprintf(team[0].name, sizeof team[0].name, "%s", "Ada L.");
    int best = 0;
    for (int i = 1; i < 3; i++) {
        if (team[i].score > team[best].score) {
            best = i;
        }
    }
    for (int i = 0; i < 3; i++) {
        printf("%-8s %3d%s\n", team[i].name, team[i].score, i == best ? "  <- best" : "");
    }
    return 0;
}

🧪 **Try it:** try `team[0].name = "Bob";` and read the error: an array can't be assigned. Then add a fourth player.

## 5. enum: naming a set of choices

An **enum** gives names to a fixed set of choices. Behind the names are ints (0, 1, 2… by default), but the names make the code readable and work well in a `switch`.

In [ ]:
#include <stdio.h>

typedef enum { RED, YELLOW, GREEN } Light;

Light next(Light l) {
    switch (l) {
        case RED:
            return GREEN;
        case GREEN:
            return YELLOW;
        case YELLOW:
            return RED;
    }
    return RED;
}

int main(void) {
    const char *names[] = {"red", "yellow", "green"};
    Light l = RED;
    for (int i = 0; i < 4; i++) {
        printf("%s (%d)\n", names[l], l);
        l = next(l);
    }
    return 0;
}

🧪 **Try it:** delete the `case YELLOW:` lines and read the warning: the switch doesn't handle every value of the enum.

## 6. sizeof and padding

A struct's size can be **more** than the sum of its fields: the compiler may add unused bytes (**padding**) so that each field starts at an address the processor reads quickly. Ordering the fields from biggest to smallest often reduces the padding.

In [ ]:
#include <stdio.h>

typedef struct {
    char a;
    double b;
    char c;
} Loose;

typedef struct {
    double b;
    char a;
    char c;
} Tight;

int main(void) {
    printf("fields: %zu bytes\n", sizeof(char) + sizeof(double) + sizeof(char));
    printf("Loose: %zu bytes, Tight: %zu bytes\n", sizeof(Loose), sizeof(Tight));
    return 0;
}

🧪 **Try it:** add an `int` field to each struct, in different places, and compare the sizes.

## 7. Comparing structs

`==` doesn't work on structs: the code doesn't compile. Write a function that compares field by field (and with `strcmp` for text fields). Don't compare structs byte by byte with `memcmp`: the padding bytes may hold garbage.

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

int same_point(Point a, Point b) {
    return a.x == b.x && a.y == b.y;
}

int main(void) {
    Point a = {1, 2};
    Point b = {1, 2};
    Point c = {2, 1};
    printf("a and b: %d, a and c: %d\n", same_point(a, b), same_point(a, c));
    return 0;
}

🧪 **Try it:** write `a == b` in the `printf` and read the error.

## In short

A **struct** groups values of different types under one name, like a Java class with only fields. `typedef` gives the type a short name, and an `enum` names a fixed set of choices.

```c
#include <stdio.h>

typedef enum { ACTIVE, FROZEN } Status;

typedef struct {
    char owner[21];
    long balance;            // in cents
    Status status;
} Account;

void deposit(Account *a, long cents) {   // a pointer: changes the caller's struct
    if (a->status == ACTIVE) {
        a->balance += cents;               // a->balance is (*a).balance
    }
}

int main(void) {
    Account accounts[2] = {{"Ada", 1000, ACTIVE}, {"Alan", 500, FROZEN}};
    deposit(&accounts[0], 250);
    printf("%s: %ld\n", accounts[0].owner, accounts[0].balance);   // Ada: 1250
    return 0;
}
```

Assigning or passing a struct **copies every field**, arrays inside it included, so functions that change a struct, or work on a big one, take a pointer.

**Watch out:** `==` doesn't work on structs. Compare them field by field.

## Readings

- [Structs](https://beej.us/guide/bgc/html/split/structs.html) (Beej's Guide to C Programming, chapter 8, 8 min): Declaring a struct, initializers, passing a struct to a function (and why a pointer is usual), the arrow operator, copying, and why == can't compare structs.
- [Structures and enumerated types](https://www.freecodecamp.org/news/the-c-beginners-handbook/#heading-structures) (freeCodeCamp, The C Beginner's Handbook, 5 min): Optional: structs with typedef, then "Enumerated types" just before it, for the enum used in Bank Ledger.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Fractions** · Easy
- **Student Records** · Medium
- **Bank Ledger** · Medium
- **Point in Polygon** · Hard